In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss, f1_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder

In [ ]:
train = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/train.csv')
test = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/test.csv')
greeks = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/greeks.csv')
sample_sub = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/sample_submission.csv')

In [ ]:
encoder = LabelEncoder()
train["EJ"] = encoder.fit_transform(train["EJ"])
test["EJ"] = encoder.fit_transform(test["EJ"])
train = train.dropna().reset_index(drop=True)

In [ ]:
train['kfold'] = -1

kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_indicies, valid_indicies) in enumerate(kf.split(X=train)):
    train.loc[valid_indicies, "kfold"] = fold
features_cols = train.columns[1:-2]
label    = train.columns[-2]

In [ ]:
import lightgbm as lgb
final_valid_predictions = {}
final_test_predictions = []
scores = []
s = []

for k in range(5):

    train_df = train[train['kfold'] !=k].reset_index(drop=True)
    val_df = train[train['kfold'] ==k].reset_index(drop=True)
    valid_ids = val_df.Id.values.tolist()

    X_train = train_df[features_cols].values
    y_train = train_df[label].values
    X_val = val_df[features_cols].values
    y_val = val_df[label].values

    model = lgb.LGBMClassifier()
    model.fit(X_train, y_train)

    preds_valid = model.predict_proba(X_val)
    preds_test  = model.predict_proba(test[features_cols].values)
    final_test_predictions.append(preds_test)
    final_valid_predictions.update(dict(zip(valid_ids, preds_valid)))
    logloss = log_loss(y_val, preds_valid)

    s.append(logloss)
    print(k, logloss)
print(s)
print(np.mean(s), np.std(s))


In [ ]:
final_valid_predictions = pd.DataFrame.from_dict(final_valid_predictions, orient="index").reset_index()
final_valid_predictions.columns = ['Id', 'class_0', 'class_1']
final_valid_predictions.to_csv(r"oof.csv", index=False)

final_test_predictions = (final_test_predictions[0] + final_test_predictions[1] + final_test_predictions[2] + final_test_predictions[3] + final_test_predictions[4])/5
test_dict = {}
test_dict.update(dict(zip(test.Id.values.tolist(), final_test_predictions)))
submission = pd.DataFrame.from_dict(test_dict, orient="index").reset_index()
submission.columns = ['Id', 'class_0', 'class_1']                       

submission.to_csv(r"submission.csv", index=False)
submission